# 🌽 Maize Crop Detection — YOLOv9m + CBAM
**Project:** GoPro-based maize crop detection across Nigerian states
**Authors:** Nivas Panidapu, Chinoye Agadi
**Mentor:** Dr. Kyle Davis, University of Delaware

## Pipeline Overview
1. `Cell 1` — Install dependencies
2. `Cell 2` — GPU check
3. `Cell 3` — Download dataset from Roboflow
4. `Cell 4` — Download YOLOv9m pretrained weights
5. `Cell 5` — Configure paths and training parameters
6. `Cell 6` — Create CBAM model architecture
7. `Cell 7` — Train model (`switch_train = True`)
8. `Cell 8` — Run inference on test images (`switch_pred = True`)
9. `Cell 9` — Match detections to GPS coordinates
10. `Cell 10` — Save results to Excel
11. `Cell 11` — Generate interactive map
12. `Cell 12` — Visualize best detections
13. `Cell 13` — Download weights and results

> **Before running:** Add `ROBOFLOW_API_KEY` to Colab Secrets (🔑 icon in left sidebar)
> Upload `coord2025_all.csv` to `/content/` before running inference cells

In [ ]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
!pip install -q opencv-python ultralytics pandas pillow pyproj roboflow folium openpyxl
!pip install -q numpy==1.24.4
print('✅ All dependencies installed')


In [ ]:
# ============================================================
# CELL 2 — GPU Check
# Requires GPU runtime: Runtime → Change runtime type → GPU
# ============================================================
import torch

if torch.cuda.is_available():
    print('✅ GPU:', torch.cuda.get_device_name(0))
    print('   VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('⚠️  No GPU detected')
    print('   Go to: Runtime → Change runtime type → GPU')


In [ ]:
# ============================================================
# CELL 3 — Download Dataset from Roboflow
# Dataset: crop-detection-tigp8 (v24)
# Format:  YOLOv9 | Split: 81/9/10 | Class: Maize
# ============================================================
import os
from roboflow import Roboflow

# Load API key securely from Colab Secrets
try:
    from google.colab import userdata
    api_key = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    api_key = os.environ.get('ROBOFLOW_API_KEY', '')

if not api_key:
    raise ValueError('ROBOFLOW_API_KEY not set. Add it to Colab Secrets (🔑 icon).')

rf      = Roboflow(api_key=api_key)
project = rf.workspace('chinoye').project('crop-detection-tigp8')
version = project.version(24)  # Final dataset — Maize class, no resize
dataset = version.download('yolov9')
print('✅ Dataset downloaded to:', dataset.location)


In [ ]:
# ============================================================
# CELL 4 — Download YOLOv9m Pretrained Weights
# These weights are used as a starting point for fine-tuning
# ============================================================
import subprocess, os

result = subprocess.run(
    ['wget', '-q',
     'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov9m.pt',
     '-O', 'yolov9m.pt'],
    capture_output=True
)

if os.path.exists('yolov9m.pt'):
    print('✅ yolov9m.pt ready')
else:
    # Fallback download method
    import urllib.request
    urllib.request.urlretrieve(
        'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov9m.pt',
        'yolov9m.pt'
    )
    print('✅ yolov9m.pt downloaded via urllib')


In [ ]:
# ============================================================
# CELL 5 — Configuration
# Update GPS_CSV path if stored elsewhere
# ============================================================
import os

# ── Paths ──────────────────────────────────────────────────
ROOT     = dataset.location          # Root dataset folder
DATA_DIR = dataset.location          # Dataset directory
TEST_DIR = f'{DATA_DIR}/test/images' # Test images for inference

project_dir = f'{ROOT}/runs'                          # Training output folder
name        = 'maize_v24_cbam'                        # Experiment name
MODEL_DIR   = f'{project_dir}/{name}/weights/best.pt' # Best model weights path
PRED_DIR    = f'{ROOT}/predictions'                   # Inference output folder

# GPS CSV — field survey coordinates (1.6M entries across 8 Nigerian states)
# Upload coord2025_all.csv to /content/ before running inference
GPS_CSV = '/content/coord2025_all.csv'

# ── Training Parameters ────────────────────────────────────
epochs        = 200          # Max training epochs
device        = 0            # GPU device ID
optimizer     = 'AdamW'      # AdamW optimizer — better than SGD for this dataset
lr0           = 0.001        # Initial learning rate
lrf           = 0.01         # Final learning rate ratio
warmup_epochs = 5            # Gradual LR warmup to avoid early instability
cos_lr        = True         # Cosine LR schedule
patience      = 40           # Early stopping patience (epochs without improvement)
batch         = 16           # Batch size — reduce to 8 if out of memory
imgsz         = 640          # Training image size (native YOLO resolution)
single_cls    = True         # Single class detection (Maize only)
save_period   = 10           # Save checkpoint every N epochs

print(f'Dataset     : {DATA_DIR}')
print(f'Test images : {TEST_DIR}')
print(f'Model dir   : {MODEL_DIR}')
print(f'GPS CSV     : {GPS_CSV}')
print(f'imgsz       : {imgsz} | batch: {batch} | patience: {patience}')


In [ ]:
# ============================================================
# CELL 6 — Create CBAM Model Architecture Config
# CBAM (Convolutional Block Attention Module) adds:
#   - Channel Attention: which features matter (maize vs soil)
#   - Spatial Attention: where to look in the image
# Applied at P3, P4, P5 backbone stages
# ============================================================
cbam_yaml = '''
# YOLOv9m + CBAM Attention
# Single class: Maize
nc: 1
scales:
  m: [0.75, 0.75, 768]

backbone:
  - [-1, 1, Conv,  [64,  3, 2]]        # P1 stem
  - [-1, 1, Conv,  [128, 3, 2]]        # P2
  - [-1, 3, C2f,  [128, True]]
  - [-1, 1, Conv,  [256, 3, 2]]        # P3/8
  - [-1, 6, C2f,  [256, True]]
  - [-1, 1, CBAM, [256]]               # Attention on P3
  - [-1, 1, Conv,  [512, 3, 2]]        # P4/16
  - [-1, 6, C2f,  [512, True]]
  - [-1, 1, CBAM, [512]]               # Attention on P4
  - [-1, 1, Conv,  [512, 3, 2]]        # P5/32
  - [-1, 3, C2f,  [512, True]]
  - [-1, 1, CBAM, [512]]               # Attention on P5
  - [-1, 1, SPPF, [512, 5]]            # Spatial Pyramid Pooling

head:
  - [-1,      1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 8], 1, Concat,      [1]]
  - [-1,      3, C2f,         [512]]
  - [-1,      1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 5], 1, Concat,      [1]]
  - [-1,      3, C2f,         [256]]   # P3 output
  - [-1,      1, Conv,        [256, 3, 2]]
  - [[-1,15], 1, Concat,      [1]]
  - [-1,      3, C2f,         [512]]   # P4 output
  - [-1,      1, Conv,        [512, 3, 2]]
  - [[-1,12], 1, Concat,      [1]]
  - [-1,      3, C2f,         [512]]   # P5 output
  - [[18, 21, 24], 1, Detect, [nc]]    # Detection head
'''

with open('/content/yolov9m_cbam.yaml', 'w') as f:
    f.write(cbam_yaml)
print('✅ CBAM config saved → /content/yolov9m_cbam.yaml')


In [ ]:
# ============================================================
# CELL 7 — Train Model
# Set switch_train = True  to train
# Set switch_train = False to skip (if already trained)
# ============================================================
switch_train = True

if switch_train:
    from ultralytics import YOLO

    try:
        # Load CBAM architecture with pretrained YOLOv9m weights
        model = YOLO('/content/yolov9m_cbam.yaml')
        model.load('yolov9m.pt')
        print('✅ Using YOLOv9m + CBAM architecture')
    except Exception as e:
        # Fallback to standard YOLOv9m if CBAM config fails
        print(f'⚠️  CBAM init failed: {e}')
        print('   Falling back to standard YOLOv9m')
        model = YOLO('yolov9m.pt')

    model.train(
        data          = f'{DATA_DIR}/data.yaml',
        epochs        = epochs,
        device        = device,
        optimizer     = optimizer,
        lr0           = lr0,
        lrf           = lrf,
        warmup_epochs = warmup_epochs,
        cos_lr        = cos_lr,
        patience      = patience,
        batch         = batch,
        imgsz         = imgsz,
        single_cls    = single_cls,
        save_period   = save_period,
        project       = project_dir,
        name          = name,
        dropout       = 0.1,        # Regularization — reduces overfitting
        weight_decay  = 0.001,      # L2 regularization
        mosaic        = 1.0,        # Mosaic augmentation
        mixup         = 0.15,       # MixUp augmentation
        degrees       = 15.0,       # Random rotation ±15°
        flipud        = 0.3,        # Vertical flip probability
        hsv_h         = 0.015,      # Hue augmentation
        hsv_s         = 0.7,        # Saturation augmentation
        hsv_v         = 0.4,        # Value/brightness augmentation
    )
    print('\n✅ Training complete!')
    print(f'   Best weights saved → {MODEL_DIR}')


In [ ]:
# ============================================================
# CELL 8 — Run Inference on Test Images
# Uses TTA (Test Time Augmentation) for better detection
# Strips Roboflow suffix from filenames to recover GoPro names
# Example: G0010508_JPG.rf.abc123.jpg → G0010508
# ============================================================
import os, re
import pandas as pd
from ultralytics import YOLO
from pathlib import Path

# Clear previous predictions
os.system(f'rm -rf {PRED_DIR}')
os.makedirs(PRED_DIR, exist_ok=True)


def clean_name(filename):
    """
    Extract the original GoPro image name from a Roboflow filename.
    Roboflow appends a hash suffix during upload which we strip here.

    Example:
        Input : 'G0010508_JPG.rf.97668c58f6f566675a5625282a104b49.jpg'
        Output: 'G0010508'
    """
    name = os.path.splitext(filename)[0]           # Remove .jpg
    name = re.sub(r'\.rf\.[a-f0-9]+$', '', name) # Remove .rf.<hash>
    name = re.sub(r'_[Jj][Pp][Ee]?[Gg]$', '', name) # Remove _JPG
    return name


# Load trained model
model = YOLO(MODEL_DIR)
model.model.names = {0: 'Maize'}  # Ensure correct class label in plots

# Collect all test images
img_paths = [p for p in Path(TEST_DIR).glob('*')
             if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
print(f'Running inference on {len(img_paths)} test images...')
print(f'Settings: conf=0.25, iou=0.35, TTA=True\n')

results_list = []
for img_path in img_paths:
    gopro_name = clean_name(img_path.name)

    # Run inference with TTA (augment=True applies flips/scales at inference)
    result = model(
        str(img_path),
        conf    = 0.25,   # Confidence threshold
        iou     = 0.35,   # IoU threshold for NMS
        augment = True,   # Test Time Augmentation
        verbose = False
    )[0]

    boxes = result.boxes
    if len(boxes) == 0:
        continue  # Skip images with no detections

    print(f'{gopro_name}: {len(boxes)} maize plant(s) detected')

    # Store each detection
    for i in range(len(boxes)):
        results_list.append({
            'IMG':  gopro_name,           # Original GoPro image name
            'FILE': img_path.name,        # Full Roboflow filename
            'BOX':  i + 1,                # Bounding box index
            'CONF': round(float(boxes.conf[i]), 4),        # Detection confidence
            'X1':   round(float(boxes.xyxy[i][0]), 2),     # Top-left x
            'Y1':   round(float(boxes.xyxy[i][1]), 2),     # Top-left y
            'X2':   round(float(boxes.xyxy[i][2]), 2),     # Bottom-right x
            'Y2':   round(float(boxes.xyxy[i][3]), 2),     # Bottom-right y
        })

pred_df = pd.DataFrame(results_list)
print(f'\n✅ Inference complete')
print(f'   Total detections : {len(pred_df)}')
print(f'   Unique images    : {pred_df["IMG"].nunique()}')
print(f'   Sample names     : {pred_df["IMG"].head(3).tolist()}')


In [ ]:
# ============================================================
# CELL 9 — GPS Coordinate Lookup
# Matches each detected image name to GPS coordinates from
# the field survey CSV (1,612,837 entries across 8 states)
#
# Note: We use CSV-based lookup instead of EXIF because
# Roboflow strips GPS metadata during image processing.
# ============================================================
import pandas as pd

# Verify GPS CSV exists
if not os.path.exists(GPS_CSV):
    raise FileNotFoundError(
        f'GPS CSV not found at {GPS_CSV}\n'
        'Please upload coord2025_all.csv to /content/'
    )

print('Loading GPS CSV...')
gps_df = pd.read_csv(GPS_CSV)

# Build lookup dict: {image_name: {Latitude, Longitude, State}}
# drop_duplicates keeps first occurrence when same name appears in multiple states
gps_lookup = (
    gps_df
    .drop_duplicates('IMG')
    .set_index('IMG')[['Latitude', 'Longitude', 'State']]
    .to_dict('index')
)
print(f'✅ GPS CSV loaded: {len(gps_df):,} rows | {len(gps_lookup):,} unique images')

# Match each detection to its GPS coordinates
matched   = []
unmatched = []

for _, row in pred_df.iterrows():
    img_name = row['IMG']
    if img_name in gps_lookup:
        gps = gps_lookup[img_name]
        matched.append({
            'IMG':        img_name,
            'FILE':       row['FILE'],
            'BOX':        row['BOX'],
            'CONF':       row['CONF'],
            'X1':         row['X1'],
            'Y1':         row['Y1'],
            'X2':         row['X2'],
            'Y2':         row['Y2'],
            'Latitude':   gps['Latitude'],
            'Longitude':  gps['Longitude'],
            'State':      gps['State'],
            'Class':      'Maize',
            'Confidence': row['CONF']
        })
    else:
        unmatched.append(img_name)

results_gps = pd.DataFrame(matched)
print(f'\n✅ GPS matched   : {len(results_gps)} detections')
print(f'⚠️  GPS unmatched : {len(unmatched)} detections')
if len(results_gps) > 0:
    print(f'\nSample results:')
    print(results_gps[['IMG', 'Latitude', 'Longitude', 'State', 'Confidence']].head())


In [ ]:
# ============================================================
# CELL 10 — Save Detection Results to Excel
# Output: maize_detections.xlsx
# Columns: IMG, FILE, BOX, CONF, X1, Y1, X2, Y2,
#          Latitude, Longitude, State, Class, Confidence
# ============================================================
output_excel = '/content/maize_detections.xlsx'

if len(results_gps) == 0:
    print('⚠️  No matched detections to save.')
else:
    results_gps.to_excel(output_excel, index=False)
    print(f'✅ Results saved → {output_excel}')
    print(f'   Total detections : {len(results_gps)}')
    print(f'   Unique images    : {results_gps["IMG"].nunique()}')
    print(f'   States covered   : {results_gps["State"].unique().tolist()}')

    from google.colab import files
    files.download(output_excel)


In [ ]:
# ============================================================
# CELL 11 — Generate Interactive Folium Map
# Features:
#   - Street / Dark / Satellite tile layers
#   - Density heatmap of maize detections
#   - Per-state toggleable marker layers
#   - Marker size proportional to plant count per image
#   - Popup with image name, state, confidence, coordinates
#   - Summary legend with total images and plants
# ============================================================
import folium
from folium.plugins import HeatMap, MiniMap, Fullscreen

if len(results_gps) == 0:
    print('⚠️  No data to map.')
else:
    # One marker per unique image location
    map_df = results_gps.drop_duplicates('IMG').copy()

    # Color per Nigerian state
    state_colors = {
        'Plateau':  '#2ecc71',
        'Oyo':      '#3498db',
        'Ogun':     '#e67e22',
        'Niger':    '#e74c3c',
        'Nasarawa': '#9b59b6',
        'Kwara':    '#1abc9c',
        'FCT':      '#34495e',
        'Benue':    '#c0392b'
    }

    # Centre map on mean coordinates
    centre = [map_df['Latitude'].mean(), map_df['Longitude'].mean()]
    m = folium.Map(location=centre, zoom_start=7, tiles='CartoDB positron')

    # ── Tile layers ────────────────────────────────────────
    folium.TileLayer('CartoDB positron',    name='Street Map').add_to(m)
    folium.TileLayer('CartoDB dark_matter', name='Dark Map').add_to(m)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Satellite', overlay=False
    ).add_to(m)

    # ── Density heatmap ────────────────────────────────────
    heat_data = [[row['Latitude'], row['Longitude']] for _, row in map_df.iterrows()]
    HeatMap(
        heat_data, name='Density Heatmap',
        min_opacity=0.4, radius=25, blur=15,
        gradient={0.2:'blue', 0.4:'lime', 0.6:'yellow', 0.8:'orange', 1.0:'red'}
    ).add_to(m)

    # ── Per-state feature groups (toggleable in layer control) ─
    state_groups = {}
    for state in map_df['State'].unique():
        fg = folium.FeatureGroup(name=f'📍 {state}', show=True)
        state_groups[state] = fg
        m.add_child(fg)

    # ── Detection markers ──────────────────────────────────
    for _, row in map_df.iterrows():
        state     = row['State']
        hex_color = state_colors.get(state, '#2ecc71')

        # Count detections and average confidence for this image
        det_count = len(results_gps[results_gps['IMG'] == row['IMG']])
        avg_conf  = results_gps[results_gps['IMG'] == row['IMG']]['Confidence'].mean()

        # Marker radius scales with number of plants detected
        radius = min(5 + det_count, 15)

        popup_html = f"""
        <div style='font-family:Arial; min-width:200px;'>
            <div style='background:{hex_color}; color:white; padding:6px 10px;
                        border-radius:4px 4px 0 0; font-weight:bold;'>🌽 Maize Detection</div>
            <div style='padding:8px 10px; border:1px solid #ddd;
                        border-top:none; border-radius:0 0 4px 4px; font-size:12px;'>
                <b>Image:</b> {row['IMG']}<br>
                <b>State:</b> {state}<br>
                <b>Maize Plants:</b> {det_count}<br>
                <b>Avg Confidence:</b> {avg_conf:.0%}<br>
                <b>Latitude:</b> {row['Latitude']:.6f}<br>
                <b>Longitude:</b> {row['Longitude']:.6f}
            </div>
        </div>"""

        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=radius,
            color='white', weight=2,
            fill=True, fill_color=hex_color, fill_opacity=0.85,
            popup=folium.Popup(popup_html, max_width=230),
            tooltip=f'🌽 {state} | {det_count} plants | {avg_conf:.0%} conf'
        ).add_to(state_groups[state])

    # ── Summary legend ─────────────────────────────────────
    state_counts     = map_df['State'].value_counts().to_dict()
    total_images     = len(map_df)
    total_detections = len(results_gps)

    stats_rows = ''.join([
        f"<tr><td style='padding:3px 8px;'>"
        f"<span style='color:{state_colors.get(s,'#2ecc71')}; font-size:16px;'>●</span> {s}</td>"
        f"<td style='padding:3px 8px; text-align:right;'>{c}</td></tr>"
        for s, c in sorted(state_counts.items(), key=lambda x: -x[1])
    ])

    legend_html = f"""
    <div style='position:fixed; bottom:30px; left:30px; z-index:1000;
                background:white; padding:15px; border-radius:8px;
                box-shadow:0 2px 12px rgba(0,0,0,0.2); font-family:Arial; min-width:200px;'>
        <div style='font-weight:bold; font-size:14px; margin-bottom:10px;
                    border-bottom:2px solid #2ecc71; padding-bottom:6px;'>
            🌽 Maize Detection Summary
        </div>
        <div style='font-size:11px; color:#666; margin-bottom:8px;'>
            Each marker = 1 GoPro image location<br>
            Marker size = number of plants detected
        </div>
        <table style='width:100%; font-size:12px;'>
            <tr style='background:#f8f8f8; font-weight:bold;'>
                <td style='padding:3px 8px;'>State</td>
                <td style='padding:3px 8px; text-align:right;'>Images</td>
            </tr>
            {stats_rows}
            <tr style='border-top:2px solid #eee;'>
                <td style='padding:5px 8px; font-weight:bold;'>Total Images</td>
                <td style='padding:5px 8px; text-align:right; font-weight:bold;'>{total_images}</td>
            </tr>
            <tr style='background:#f0fff0;'>
                <td style='padding:5px 8px; font-weight:bold; color:#2ecc71;'>Total Plants</td>
                <td style='padding:5px 8px; text-align:right; font-weight:bold; color:#2ecc71;'>{total_detections}</td>
            </tr>
        </table>
        <div style='font-size:10px; color:#aaa; margin-top:8px;'>
            Model: YOLOv9m+CBAM | Dataset: Roboflow v24
        </div>
    </div>"""
    m.get_root().html.add_child(folium.Element(legend_html))

    # ── Title bar ──────────────────────────────────────────
    title_html = """
    <div style='position:fixed; top:15px; left:50%; transform:translateX(-50%);
                z-index:1000; background:rgba(255,255,255,0.95);
                padding:10px 24px; border-radius:8px;
                box-shadow:0 2px 12px rgba(0,0,0,0.2); font-family:Arial; text-align:center;'>
        <div style='font-size:16px; font-weight:bold; color:#2c3e50;'>
            🌽 Maize Crop Detection — Nigeria Field Survey 2025
        </div>
        <div style='font-size:11px; color:#7f8c8d; margin-top:3px;'>
            GoPro Imagery · YOLOv9m+CBAM · GPS Coordinates from Field Survey
        </div>
    </div>"""
    m.get_root().html.add_child(folium.Element(title_html))

    # ── Map plugins ────────────────────────────────────────
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)

    # ── Save and download ──────────────────────────────────
    map_path = '/content/maize_detection_map.html'
    m.save(map_path)
    print(f'✅ Map saved → {map_path}')
    print(f'   GoPro images mapped  : {total_images}')
    print(f'   Maize plants detected: {total_detections}')
    print(f'   States covered       : {list(state_counts.keys())}')

    from google.colab import files
    files.download(map_path)


In [ ]:
# ============================================================
# CELL 12 — Visualize Best Detection Images
# Shows the 2 test images with highest average confidence
# ============================================================
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Find top 2 images by average confidence
best_images = (
    pred_df.groupby('IMG')['CONF']
    .mean()
    .sort_values(ascending=False)
    .head(2)
    .index.tolist()
)
print(f'Top 2 images by confidence: {best_images}')

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

for idx, img_name in enumerate(best_images):
    # Find matching file in test folder
    matches = [p for p in Path(TEST_DIR).glob('*')
               if clean_name(p.name) == img_name]
    if not matches:
        print(f'⚠️  Image not found: {img_name}')
        continue

    # Run inference and plot boxes
    result     = model(str(matches[0]), conf=0.25, iou=0.35, augment=True)[0]
    img_plotted = result.plot()
    img_rgb    = cv2.cvtColor(img_plotted, cv2.COLOR_BGR2RGB)

    det_count = len(pred_df[pred_df['IMG'] == img_name])
    avg_conf  = pred_df[pred_df['IMG'] == img_name]['CONF'].mean()

    axes[idx].imshow(img_rgb)
    axes[idx].set_title(
        f'{img_name}\n{det_count} detections | Avg Confidence: {avg_conf:.0%}',
        fontsize=12, fontweight='bold'
    )
    axes[idx].axis('off')

plt.suptitle('Best Maize Detections — YOLOv9m + CBAM', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/best_detections.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → /content/best_detections.png')

from google.colab import files
files.download('/content/best_detections.png')


In [ ]:
# ============================================================
# CELL 13 — Download Model Weights and Results
# ============================================================
from google.colab import files
import shutil

# Download best model weights
print('Downloading model weights...')
files.download(MODEL_DIR)

# Download all results as a zip archive
print('Downloading results zip...')
shutil.make_archive('/content/maize_results', 'zip', ROOT)
files.download('/content/maize_results.zip')

print('✅ All files downloaded')
